# TP : Active Learning Avancé — Dataset UCI HAR (Kaggle)
**Stratégies :** S1 (Random), S14 (Dropout Committee), S28 (Hybride Self-Adaptive)  
**Objectif :** Annoter intelligemment un minimum d'exemples pour maximiser l'accuracy avec un budget limité.
---
## Pseudocode de la boucle Active Learning
```
ENTRÉE : Pool non étiqueté U, petit ensemble étiqueté L0, budget B, classifieur f
POUR t = 0 À T-1 :
    f.entrainer(Lt)
    POUR chaque x dans Ut : calculer score(x, f)
    S* = sélectionner_top_k(score)
    labels = oracle.interroger(S*)
    Lt+1 = Lt ∪ (S*, labels) ; Ut+1 = Ut \ S*
    mettre_à_jour_paramètre_adaptatif()  # seulement pour S28
FIN POUR
```

In [5]:
pip install numpy pandas matplotlib seaborn scikit-learn torch requests

Defaulting to user installation because normal site-packages is not writeable
  Using cached torch-2.11.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached cuda_bindings-13.2.0-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.3 kB)
  Using cached nvidia_cudnn_cu13-9.19.0.56-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.0-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.28.9-py3-none-manylinux_2_18_x86_64.whl.metadata (2.0 kB)
  Using cached nvidia_nvshmem_cu13-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.1

In [6]:
pip install --default-timeout=200 torch torchvision --index-url https://download.pytorch.org/whl/cu124

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached https://download-r2.pytorch.org/whl/cu124/torchvision-0.21.0%2Bcu124-cp311-cp311-linux_x86_64.whl.metadata (6.1 kB)
  Using cached https://download-r2.pytorch.org/whl/cu124/torch-2.6.0%2Bcu124-cp311-cp311-linux_x86_64.whl.metadata (28 kB)
  Using cached https://download.pytorch.org/whl/cu124/nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl (24.6 MB)
  Using cached https://download.pytorch.org/whl/cu124/nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl (883 kB)
  Using cached https://download.pytorch.org/whl/cu124/nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl (13.8 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 6.4 MB/s eta 0:00:00m eta 0:00:010:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 9.8 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━

In [7]:
# ==================== CELLULE 1 : IMPORTS & CONFIGURATION ====================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, json, os, requests, zipfile, io
from pathlib import Path
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
%matplotlib inline

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             silhouette_score, adjusted_rand_score)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from scipy.spatial.distance import cdist

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device utilisé : {DEVICE}')

Device utilisé : cuda


## 1. Téléchargement et chargement du dataset UCI HAR

In [8]:
# ==================== CELLULE 2 : CHARGEMENT DU DATASET ====================
# Téléchargement du dataset UCI HAR depuis Kaggle
def download_har_dataset():
    """Télécharge et extrait le dataset UCI HAR."""
    # URL du dataset UCI HAR (version Kaggle)
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip"
    data_dir = Path('UCI_HAR_Dataset')
    
    if not data_dir.exists():
        print('Téléchargement du dataset UCI HAR...')
        r = requests.get(url)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        z.extractall('.')
        print('Extraction terminée.')
    else:
        print('Dataset déjà présent.')
    return data_dir

data_dir = download_har_dataset()

# Chargement des données
def load_har_data(data_dir):
    """Charge les données UCI HAR et retourne X_train, X_test, y_train, y_test."""
    base = Path(data_dir) / 'UCI HAR Dataset'
    
    # Chargement des features
    X_train = pd.read_csv(base / 'train' / 'X_train.txt', sep=r'\s+', header=None).values
    X_test = pd.read_csv(base / 'test' / 'X_test.txt', sep=r'\s+', header=None).values
    
    # Chargement des labels
    y_train = pd.read_csv(base / 'train' / 'y_train.txt', sep=r'\s+', header=None).values.ravel()
    y_test = pd.read_csv(base / 'test' / 'y_test.txt', sep=r'\s+', header=None).values.ravel()
    
    # Noms des features
    features = pd.read_csv(base / 'features.txt', sep=r'\s+', header=None)[1].values
    
    # Noms des activités
    activities = pd.read_csv(base / 'activity_labels.txt', sep=r'\s+', header=None)[1].values
    
    print(f'X_train : {X_train.shape}, X_test : {X_test.shape}')
    print(f'Activités : {activities}')
    
    return X_train, X_test, y_train, y_test, features, activities

X_train_full, X_test, y_train_full, y_test, feature_names, activity_labels = load_har_data(data_dir)

# Mapping des labels
label_map = {1:'WALKING', 2:'WALKING_UP', 3:'WALKING_DOWN', 4:'SITTING', 5:'STANDING', 6:'LAYING'}
y_train_full_str = np.array([label_map[l] for l in y_train_full])
y_test_str = np.array([label_map[l] for l in y_test])

Téléchargement du dataset UCI HAR...
Extraction terminée.


FileNotFoundError: [Errno 2] No such file or directory: 'UCI_HAR_Dataset/UCI HAR Dataset/train/X_train.txt'

## 2. Analyse Exploratoire des Données (EDA)

In [ ]:
# ==================== CELLULE 3 : EDA ====================
# 1. Distribution des classes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
unique_train, counts_train = np.unique(y_train_full, return_counts=True)
unique_test, counts_test = np.unique(y_test, return_counts=True)
axes[0].bar([label_map[u] for u in unique_train], counts_train, color='steelblue')
axes[0].set_title('Distribution des classes — Entraînement')
axes[0].tick_params(axis='x', rotation=45)
axes[1].bar([label_map[u] for u in unique_test], counts_test, color='coral')
axes[1].set_title('Distribution des classes — Test')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# 2. Statistiques descriptives
df_train = pd.DataFrame(X_train_full[:, :10], columns=[f'f{i}' for i in range(10)])
print('Statistiques descriptives (10 premières features) :')
display(df_train.describe())

# 3. Matrice de corrélation (subset)
fig, ax = plt.subplots(figsize=(10, 8))
corr = df_train.corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', ax=ax)
ax.set_title('Matrice de corrélation (10 premières features)')
plt.show()

# 4. Vérification des valeurs manquantes
print(f'Valeurs manquantes dans X_train_full : {np.isnan(X_train_full).sum()}')
print(f'Valeurs manquantes dans X_test : {np.isnan(X_test).sum()}')

## 3. Standardisation et Réduction de Dimension (PCA / t-SNE)

In [ ]:
# ==================== CELLULE 4 : STANDARDISATION + PCA + t-SNE ====================
# Standardisation
scaler = StandardScaler()
X_train_full_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

# PCA
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_train_full_scaled)
print(f'Variance expliquée PCA (2D) : {pca.explained_variance_ratio_.sum():.2%}')

plt.figure(figsize=(8, 6))
for label in np.unique(y_train_full):
    mask = y_train_full == label
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=label_map[label], alpha=0.5, s=10)
plt.legend(markerscale=3)
plt.title('PCA 2D — UCI HAR (entraînement)')
plt.tight_layout()
plt.show()

# t-SNE (sur un sous-ensemble pour accélérer)
n_tsne = 2000
idx_tsne = np.random.choice(len(X_train_full_scaled), size=n_tsne, replace=False)
tsne = TSNE(n_components=2, random_state=SEED, perplexity=50)
X_tsne = tsne.fit_transform(X_train_full_scaled[idx_tsne])

plt.figure(figsize=(8, 6))
for label in np.unique(y_train_full):
    mask = y_train_full[idx_tsne] == label
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=label_map[label], alpha=0.6, s=15)
plt.legend(markerscale=3)
plt.title(f't-SNE 2D — UCI HAR (sous-ensemble {n_tsne} points)')
plt.tight_layout()
plt.show()

## 4. Clustering (K-Means) et Évaluation

In [ ]:
# ==================== CELLULE 5 : CLUSTERING ====================
kmeans = KMeans(n_clusters=6, random_state=SEED, n_init=10)
clusters = kmeans.fit_predict(X_train_full_scaled)

# Scores
sil_score = silhouette_score(X_train_full_scaled, clusters)
ari_score = adjusted_rand_score(y_train_full, clusters)
print(f'Silhouette Score : {sil_score:.4f}')
print(f'Adjusted Rand Index : {ari_score:.4f}')

# Matrice de contingence
contingency = pd.crosstab([label_map[l] for l in y_train_full],
                          clusters, rownames=['Vraie classe'], colnames=['Cluster'])
display(contingency)

# Visualisation des clusters sur PCA
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for label in np.unique(y_train_full):
    mask = y_train_full == label
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=label_map[label], alpha=0.4, s=10)
axes[0].set_title('Vraies classes')
axes[0].legend(markerscale=3)
scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='tab10', alpha=0.5, s=10)
axes[1].set_title('Clusters K-Means')
fig.colorbar(scatter, ax=axes[1])
plt.tight_layout()
plt.show()

## 5. Préparation pour l'Active Learning

In [ ]:
# ==================== CELLULE 6 : PRÉPARATION ACTIVE LEARNING ====================
# On utilise tout l'ensemble d'entraînement comme pool.
# On garde l'ensemble de test original pour l'évaluation.
X_pool = X_train_full_scaled.copy()
y_pool = y_train_full.copy()

# Échantillon initial étiqueté : 3 exemples par classe
n_init_per_class = 3
L_idx = []
for c in np.unique(y_pool):
    idx_c = np.where(y_pool == c)[0]
    L_idx.extend(np.random.choice(idx_c, size=n_init_per_class, replace=False))
L_idx = np.array(L_idx)
U_idx = np.setdiff1d(np.arange(len(y_pool)), L_idx)

X_L, y_L = X_pool[L_idx], y_pool[L_idx]
X_U, y_U = X_pool[U_idx], y_pool[U_idx]

print(f'Ensemble étiqueté initial : {len(X_L)} exemples')
print(f'Pool non étiqueté : {len(X_U)} exemples')
print(f'Ensemble de test : {len(X_test_scaled)} exemples')
print(f'Budget total disponible : {len(X_U)} annotations possibles')

## 6. Modèles et Stratégies d'Active Learning

In [ ]:
# ==================== CELLULE 7 : MODÈLES ====================

# --- Modèle RandomForest (pour S1 et S28) ---
def get_rf_model():
    return RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)

# --- Modèle MLP avec Dropout (pour S14) ---
class MLP_Dropout(nn.Module):
    def __init__(self, input_dim, hidden=64, output_dim=6, dropout=0.5):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.drop = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden, hidden // 2)
        self.bn2 = nn.BatchNorm1d(hidden // 2)
        self.fc3 = nn.Linear(hidden // 2, output_dim)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.drop(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.drop(x)
        x = self.fc3(x)
        return x

def train_mlp(model, X, y, epochs=100, batch_size=32, lr=1e-3, verbose=False):
    model.train()
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32),
                            torch.tensor(y, dtype=torch.long))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        total_loss = 0
        for bx, by in loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if verbose and epoch % 20 == 0:
            print(f'  Epoch {epoch}, loss={total_loss/len(loader):.4f}')
    return model

# --- Prédiction MC Dropout ---
def mc_predict(model, X, n_samples=25):
    model.train()  # Active le dropout
    X_t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    probs = []
    with torch.no_grad():
        for _ in range(n_samples):
            logits = model(X_t)
            probs.append(F.softmax(logits, dim=1).cpu().numpy())
    probs = np.stack(probs, axis=0)
    mean_probs = probs.mean(axis=0)
    entropy = -np.sum(mean_probs * np.log(mean_probs + 1e-10), axis=1)
    expected_entropy = -np.mean(np.sum(probs * np.log(probs + 1e-10), axis=2), axis=0)
    bald = entropy - expected_entropy
    return entropy, bald

In [ ]:
# ==================== CELLULE 8 : STRATÉGIES DE SÉLECTION ====================

# --- S1 : Random Sampling ---
def random_select(X_U, batch_size):
    return np.random.choice(len(X_U), size=batch_size, replace=False)

# --- S14 : Dropout Committee (BALD) ---
def dropout_committee_select(model, X_U, batch_size, n_dropout=25):
    _, bald = mc_predict(model, X_U, n_samples=n_dropout)
    return np.argsort(bald)[-batch_size:]

# --- S28 : Hybrid Self-Adaptive ---
def hybrid_select(model, X_L, y_L, X_U, alpha, beta, gamma, batch_size):
    """Sélection hybride : incertitude + diversité + représentativité."""
    # Incertitude (entropie)
    if isinstance(model, RandomForestClassifier):
        probs = model.predict_proba(X_U)
        probs = np.clip(probs, 1e-10, 1.0)
        U = -np.sum(probs * np.log(probs), axis=1)
    else:
        model.eval()
        with torch.no_grad():
            logits = model(torch.tensor(X_U, dtype=torch.float32).to(DEVICE))
            probs = F.softmax(logits, dim=1).cpu().numpy()
        U = -np.sum(probs * np.log(probs + 1e-10), axis=1)

    # Diversité : distance cosinus aux exemples déjà étiquetés
    sim_to_L = cosine_similarity(X_U, X_L)
    D = 1 - np.max(sim_to_L, axis=1)

    # Représentativité : similarité au centroïde du pool
    centroid = np.mean(X_U, axis=0, keepdims=True)
    R = cosine_similarity(X_U, centroid).flatten()

    # Normalisation min-max
    def norm(v):
        return (v - v.min()) / (v.max() - v.min() + 1e-10)
    U_n, D_n, R_n = norm(U), norm(D), norm(R)

    score = alpha * U_n + beta * D_n + gamma * R_n
    return np.argsort(score)[-batch_size:]

In [ ]:
# ==================== CELLULE 9 : CONTRÔLEUR SELF-ADAPTIVE ====================
class AdaptiveController:
    """Contrôleur adaptatif pour le poids d'exploration lambda."""
    def __init__(self, init_lambda=0.7, step=0.05, threshold=0.005):
        self.lambda_ = init_lambda
        self.step = step
        self.threshold = threshold
        self.prev_acc = None
        self.history = []

    def update(self, current_acc):
        if self.prev_acc is not None:
            delta = current_acc - self.prev_acc
            if delta < self.threshold:
                self.lambda_ = min(0.95, self.lambda_ + self.step)
            else:
                self.lambda_ = max(0.05, self.lambda_ - self.step)
        self.prev_acc = current_acc
        self.history.append(self.lambda_)
        alpha = self.lambda_
        beta = 0.7 * (1 - self.lambda_)
        gamma = 0.3 * (1 - self.lambda_)
        return alpha, beta, gamma

In [ ]:
# ==================== CELLULE 10 : BOUCLE ACTIVE LEARNING ====================
def run_active_learning(strategy, X_L, y_L, X_U, y_U, X_test, y_test,
                        batch_size=10, budget=200, adaptive=False):
    """
    Exécute la boucle d'Active Learning.
    Retourne l'historique des métriques.
    """
    X_L = X_L.copy()
    y_L = y_L.copy()
    X_U = X_U.copy()
    y_U = y_U.copy()

    n_queries = budget // batch_size
    history = {'accuracy': [], 'f1': [], 'n_labels': [], 'time': []}

    # Initialisation du modèle
    if strategy == 'S14':
        model = MLP_Dropout(input_dim=X_L.shape[1], hidden=64, output_dim=6).to(DEVICE)
        model = train_mlp(model, X_L, y_L, epochs=100, lr=1e-3, verbose=True)
        controller = None
    else:
        model = get_rf_model()
        model.fit(X_L, y_L)
        controller = AdaptiveController() if adaptive else None

    for t in range(n_queries):
        t0 = time.time()

        # Évaluation
        if strategy == 'S14':
            model.eval()
            X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
            with torch.no_grad():
                pred = model(X_test_t).argmax(dim=1).cpu().numpy()
            model.train()
        else:
            pred = model.predict(X_test)
        acc = accuracy_score(y_test, pred)
        f1 = f1_score(y_test, pred, average='macro')
        history['accuracy'].append(acc)
        history['f1'].append(f1)
        history['n_labels'].append(len(y_L))

        # Sélection
        if strategy == 'S1':
            selected = random_select(X_U, batch_size)
        elif strategy == 'S14':
            selected = dropout_committee_select(model, X_U, batch_size)
        elif strategy == 'S28':
            alpha, beta, gamma = controller.update(acc)
            selected = hybrid_select(model, X_L, y_L, X_U, alpha, beta, gamma, batch_size)

        # Oracle
        X_new, y_new = X_U[selected], y_U[selected]

        # Mise à jour des ensembles
        X_L = np.vstack([X_L, X_new])
        y_L = np.append(y_L, y_new)
        mask = np.ones(len(X_U), dtype=bool)
        mask[selected] = False
        X_U, y_U = X_U[mask], y_U[mask]

        # Réentraînement
        if strategy == 'S14':
            model = MLP_Dropout(input_dim=X_L.shape[1], hidden=64, output_dim=6).to(DEVICE)
            model = train_mlp(model, X_L, y_L, epochs=100, lr=1e-3)
        else:
            model.fit(X_L, y_L)

        history['time'].append(time.time() - t0)

        if t % 5 == 0:
            print(f'  Itération {t}: acc={acc:.4f}, f1={f1:.4f}, '
                  f'labels={len(y_L)}, pool restant={len(X_U)}')

    return history, model

## 7. Exécution des Expériences

In [ ]:
# ==================== CELLULE 11 : EXÉCUTION ====================
BUDGET = 200
BATCH = 10

print('\n===== S1 : Random Sampling =====')
t0 = time.time()
hist_s1, model_s1 = run_active_learning('S1', X_L, y_L, X_U, y_U, X_test_scaled, y_test,
                                        batch_size=BATCH, budget=BUDGET)
print(f'Temps total S1 : {time.time()-t0:.1f}s')

print('\n===== S14 : Dropout Committee =====')
t0 = time.time()
hist_s14, model_s14 = run_active_learning('S14', X_L, y_L, X_U, y_U, X_test_scaled, y_test,
                                          batch_size=BATCH, budget=BUDGET)
print(f'Temps total S14 : {time.time()-t0:.1f}s')

print('\n===== S28 : Hybrid Self-Adaptive =====')
t0 = time.time()
hist_s28, model_s28 = run_active_learning('S28', X_L, y_L, X_U, y_U, X_test_scaled, y_test,
                                          batch_size=BATCH, budget=BUDGET, adaptive=True)
print(f'Temps total S28 : {time.time()-t0:.1f}s')

## 8. Visualisation des Résultats

In [ ]:
# ==================== CELLULE 12 : COURBES D'APPRENTISSAGE ====================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Accuracy
axes[0].plot(hist_s1['n_labels'], hist_s1['accuracy'], 'o-', label='S1 Random', markersize=4)
axes[0].plot(hist_s14['n_labels'], hist_s14['accuracy'], 's-', label='S14 Dropout Comm.', markersize=4)
axes[0].plot(hist_s28['n_labels'], hist_s28['accuracy'], 'd-', label='S28 Hybride Adapt.', markersize=4)
axes[0].set_xlabel('Nombre d\'exemples étiquetés')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Active Learning — Accuracy vs Budget')
axes[0].legend()
axes[0].grid(True)

# F1-score
axes[1].plot(hist_s1['n_labels'], hist_s1['f1'], 'o-', label='S1 Random', markersize=4)
axes[1].plot(hist_s14['n_labels'], hist_s14['f1'], 's-', label='S14 Dropout Comm.', markersize=4)
axes[1].plot(hist_s28['n_labels'], hist_s28['f1'], 'd-', label='S28 Hybride Adapt.', markersize=4)
axes[1].set_xlabel('Nombre d\'exemples étiquetés')
axes[1].set_ylabel('F1-score (macro)')
axes[1].set_title('Active Learning — F1-score vs Budget')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ==================== CELLULE 13 : TABLEAU COMPARATIF FINAL ====================
def final_metrics(hist, name):
    return {
        'Stratégie': name,
        'Accuracy finale': hist['accuracy'][-1],
        'F1-score finale': hist['f1'][-1],
        'Nb labels utilisés': hist['n_labels'][-1],
        'Temps total (s)': sum(hist['time'])
    }

results_df = pd.DataFrame([
    final_metrics(hist_s1, 'S1 Random'),
    final_metrics(hist_s14, 'S14 Dropout Comm.'),
    final_metrics(hist_s28, 'S28 Hybride Adapt.')
])
display(results_df.round(4))

# Matrices de confusion finales
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (hist, name) in zip(axes, [(hist_s1, 'S1'), (hist_s14, 'S14'), (hist_s28, 'S28')]):
    pred = model_s1.predict(X_test_scaled) if name == 'S1' else \
           (model_s14 if name == 'S14' else model_s28).predict(X_test_scaled) if name != 'S14' else \
           model_s14(torch.tensor(X_test_scaled, dtype=torch.float32).to(DEVICE)).argmax(dim=1).cpu().numpy()
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=activity_labels, yticklabels=activity_labels)
    ax.set_title(f'Matrice de confusion — {name}')
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')
plt.tight_layout()
plt.show()

In [ ]:
# ==================== CELLULE 14 : SAUVEGARDE JSON ====================
results = {
    's1': {'accuracy': hist_s1['accuracy'], 'f1': hist_s1['f1'],
           'n_labels': hist_s1['n_labels'], 'time': hist_s1['time']},
    's14': {'accuracy': hist_s14['accuracy'], 'f1': hist_s14['f1'],
            'n_labels': hist_s14['n_labels'], 'time': hist_s14['time']},
    's28': {'accuracy': hist_s28['accuracy'], 'f1': hist_s28['f1'],
            'n_labels': hist_s28['n_labels'], 'time': hist_s28['time']},
    'config': {
        'dataset': 'UCI HAR',
        'budget': BUDGET,
        'batch_size': BATCH,
        'n_init_per_class': n_init_per_class,
        'seed': SEED
    }
}

with open('active_learning_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print('Résultats sauvegardés dans active_learning_results.json')

# Rechargement pour vérification
with open('active_learning_results.json', 'r', encoding='utf-8') as f:
    loaded = json.load(f)
print('Clés du fichier JSON :', list(loaded.keys()))
print('Accuracy finale S28 :', loaded['s28']['accuracy'][-1])

## 9. Interprétation et Conclusion

- **S1 (Random)** : référence basse, améliore lentement.
- **S14 (Dropout Committee)** : progression rapide grâce à l'incertitude, mais peut ignorer des zones denses.
- **S28 (Hybride adaptative)** : meilleure convergence grâce à l'équilibre incertitude/diversité/représentativité et à l'auto-adaptation.

**Pour exécuter ce notebook :**
1. Installez les dépendances : `pip install numpy pandas matplotlib seaborn scikit-learn torch requests`
2. Lancez Jupyter : `jupyter notebook`
3. Exécutez les cellules dans l'ordre.